# 🚀 Project 12 · vLLM Agent Inference

Serve Qwen on a Colab GPU, verify an agent reply, and measure concurrent inference.
Select **Runtime → Change runtime type → T4 GPU** before running the cells in order.

Tutorial by [Darsh Shah at freeCodeCamp](https://www.freecodecamp.org/news/how-to-scale-llm-inference-for-ai-agents-using-vllm).
See the repository README for the Colab adaptation, the recorded benchmark, and its limitations.

In [ ]:
!nvidia-smi

## 🧰 Install and verify vLLM

Run installation once per fresh Colab runtime. The demo uses a dedicated Python 3.12 environment.

In [ ]:
%%bash
set -e
python -m pip install -q uv
uv venv --python 3.12 /content/vllm-env
uv pip install --python /content/vllm-env/bin/python \
  --torch-backend=cu129 "vllm==0.12.0" openai ninja

In [ ]:
%%bash
/content/vllm-env/bin/python - <<'PYTHON'
import torch
import vllm

print("vLLM:", vllm.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU calculation:", (torch.ones(1, device="cuda") + 1).item())
PYTHON

## 🏗️ Serve Qwen

The server runs in the background on loopback. GPU kernel compilation can take time on the first start.
The server process receives the virtual environment's bin directory on PATH so FlashInfer can find Ninja.

In [ ]:
import os
import subprocess
import time
from pathlib import Path

import requests

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
models_url = "http://127.0.0.1:8000/v1/models"
env_bin = Path("/content/vllm-env/bin")
log_path = Path("/content/vllm-server.log")

env = os.environ.copy()
env["PATH"] = os.pathsep.join((str(env_bin), env.get("PATH", "")))

if "server" not in globals() or server.poll() is not None:
    subprocess.run([str(env_bin / "ninja"), "--version"], check=True, env=env)
    with log_path.open("w") as log:
        server = subprocess.Popen(
            [
                str(env_bin / "vllm"), "serve", model_id,
                "--host", "127.0.0.1",
                "--port", "8000",
                "--dtype", "half",
                "--max-model-len", "2048",
                "--enforce-eager",
            ],
            stdout=log,
            stderr=subprocess.STDOUT,
            env=env,
        )

deadline = time.monotonic() + 300
while server.poll() is None and time.monotonic() < deadline:
    try:
        response = requests.get(models_url, timeout=2)
        if response.ok and model_id in [item["id"] for item in response.json()["data"]]:
            print("Server ready. Model:", model_id)
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print("\n".join(log_path.read_text(errors="replace").splitlines()[-140:]))
    raise RuntimeError("vLLM did not become ready within the startup window.")

## 🧪 Verify an agent reply

The OpenAI client points to the loopback server. The API key value is a placeholder.

In [ ]:
%%bash
/content/vllm-env/bin/python - <<'PYTHON'
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8000/v1",
    api_key="local-demo",
    timeout=60,
)
response = client.chat.completions.create(
    model="Qwen/Qwen2.5-0.5B-Instruct",
    messages=[
        {"role": "system", "content": "You are a concise CI assistant."},
        {"role": "user", "content": "In one sentence, explain what a CI smoke test checks."},
    ],
    temperature=0,
    max_tokens=64,
)
answer = response.choices[0].message.content
if not answer or not answer.strip():
    raise RuntimeError("The model returned an empty reply.")
print("Agent reply:", answer)
PYTHON

## 📊 Extension 1 · Measure concurrent inference

Replay the same 32 requests at each concurrency setting after a four-request warm-up.
The results show batch throughput and end-to-end latency for successful HTTP requests.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from math import ceil
from statistics import median
from time import perf_counter

import requests

url = "http://127.0.0.1:8000/v1/chat/completions"
prompts = [
    "In one sentence, explain what a CI smoke test checks.",
    "In one sentence, explain what a build artifact is.",
    "In one sentence, explain why CI records test results.",
    "In one sentence, explain what a release approval checks.",
]

def send_request(prompt):
    started = perf_counter()
    try:
        reply = requests.post(
            url,
            json={
                "model": "Qwen/Qwen2.5-0.5B-Instruct",
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0,
                "max_tokens": 64,
            },
            timeout=60,
        )
        reply.raise_for_status()
        return perf_counter() - started, None
    except requests.RequestException as error:
        return None, str(error)

for prompt in prompts:
    _, error = send_request(prompt)
    if error:
        raise RuntimeError(f"Warm-up failed: {error}")

workload = prompts * 8
print(f"{'Clients':>7} {'OK':>4} {'Failed':>6} {'Req/s':>8} {'Median ms':>10} {'p95 ms':>8}")

for concurrency in (1, 4, 8):
    started = perf_counter()
    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        results = list(pool.map(send_request, workload))
    elapsed = perf_counter() - started

    latencies = sorted(latency * 1000 for latency, error in results if error is None)
    errors = [error for _, error in results if error is not None]
    median_ms = f"{median(latencies):.0f}" if latencies else "n/a"
    p95_ms = f"{latencies[ceil(0.95 * len(latencies)) - 1]:.0f}" if latencies else "n/a"

    print(
        f"{concurrency:>7} {len(latencies):>4} {len(errors):>6} "
        f"{len(latencies) / elapsed:>8.2f} {median_ms:>10} {p95_ms:>8}"
    )
    if errors:
        print("First error:", errors[0])